## The Setting

We are studying demand for a bike-sharing system.

Temperature appears to be related to the number of bikes rented, but how flexible should the prediction model be?

A simple model may fail to capture important structure. A very flexible model may fit the observed training data extremely well but perform poorly on new observations.

Our goal is therefore not simply to find the model that best fits the training data. We want a model that **generalizes**.

we focus on:

| Variable | Description |
|---|---|
| `temp` | Normalized daily temperature |
| `cnt` | Total number of bike rentals during the day |

We compare three models:

| Model | Flexibility |
|---|---|
| Linear Regression | Low |
| Polynomial Regression, Degree 9 | High, Global |
| Cubic Spline, 7 Knots | High, Local |


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
    SplineTransformer
)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

import io
import zipfile
import requests

# Bike Sharing daily data
url = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/00275/"
    "Bike-Sharing-Dataset.zip"
)

response = requests.get(url)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    bike = pd.read_csv(z.open("day.csv"))

bike = bike[["temp", "cnt"]].dropna().copy()

print("Number of days:", len(bike))
display(bike.head())

fig = px.scatter(
    bike,
    x="temp",
    y="cnt",
    labels={
        "temp": "Normalized Temperature",
        "cnt": "Daily Bike Rentals"
    },
    title="Temperature and Daily Bike Rentals"
)

fig.show()


Number of days: 731


,temp,cnt
0,0.344167,985
1,0.363478,801
2,0.196364,1349
3,0.200000,1562
4,0.226957,1600




### The direction and shape of the relationship between temperature and bike rentals 
The relationship is positive, as the temperature increases, the number of bikes rented increases too. However, the relationship changes across the temperature ranges, therefore, the points shape a curve rather than a straight line. A straight line only captures the general trend, but it would miss the underlying curved structure of the data.


###  What changes as we allow the models to become more flexible? How does the shape of the spline differ from the degree-9 polynomial?

As flexibility increases, the models fit the curve more closely. The spline model bends locally and smoothly around the knots, while the degree-9 polynomial bends a lot across the temperature range. The linear model is the simplest and least flexible of all.


### Does greater flexibility necessarily imply better prediction on new observations? 

A flexible model has more shapes available to it; therefore, it can fit the model bettter. That's why its training error is no larger than a less flexible model (We would typically expect the training error of flexible model to be lower). A greater flexibility can also make the model fit noise in the training data, that's why it does not necessarily imply better prediction.

In [2]:
X = bike[["temp"]]
y = bike["cnt"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123
)

# Linear model
lin_model = Pipeline([
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])

# Degree-9 polynomial model
poly9_model = Pipeline([
    ("scale_x", StandardScaler()),
    ("poly", PolynomialFeatures(
        degree=9,
        include_bias=False
    )),
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])

# Cubic spline with 7 knots
spline_model = Pipeline([
    ("scale_x", StandardScaler()),
    ("spline", SplineTransformer(
        n_knots=7,
        degree=3,
        include_bias=False
    )),
    ("scaler", StandardScaler()),
    ("linreg", LinearRegression())
])

# Fit all three models
lin_model.fit(X_train, y_train)
poly9_model.fit(X_train, y_train)
spline_model.fit(X_train, y_train)

models = {
    "Linear": lin_model,
    "Polynomial 9": poly9_model,
    "Spline (7 knots)": spline_model
}

# Create a grid for fitted curves
temp_grid = pd.DataFrame({
    "temp": np.linspace(
        X["temp"].min(),
        X["temp"].max(),
        200
    )
})

fig = px.scatter(
    bike,
    x="temp",
    y="cnt",
    opacity=0.5,
    labels={
        "temp": "Normalized Temperature",
        "cnt": "Daily Bike Rentals"
    },
    title="Comparing Three Models"
)

for name, model in models.items():
    fig.add_trace(
        go.Scatter(
            x=temp_grid["temp"],
            y=model.predict(temp_grid),
            mode="lines",
            name=name
        )
    )

fig.show()


# Training Error vs. Test Error

The data are divided into 80% training and 20% test.

The **training RMSE** measures how closely the model fits observations used to estimate it.

The **test RMSE** measures prediction error on observations that were not used to fit the model.


In [3]:
rows = []

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_pred)
    )

    test_rmse = np.sqrt(
        mean_squared_error(y_test, test_pred)
    )

    rows.append({
        "Model": name,
        "Training RMSE": train_rmse,
        "Test RMSE": test_rmse
    })

rmse_table = pd.DataFrame(rows)

display(rmse_table.round(2))


,Model,Training RMSE,Test RMSE
0,Linear,1477.19,1628.52
1,Polynomial 9,1386.75,1526.17
2,Spline (7 knots),1388.46,1513.27


In [7]:
rmse_long = rmse_table.melt(
    id_vars="Model",
    value_vars=[
        "Training RMSE",
        "Test RMSE"
    ],
    var_name="Dataset",
    value_name="RMSE"
)

fig = px.bar(
    rmse_long,
    x="Model",
    y="RMSE",
    color="Dataset",
    barmode="group",
    title="Training vs. Test RMSE"
)

fig.show()

### Does decreasing training error necessarily imply decreasing test error? 

No, decreasing the training error does not guarantee generalizability, it only shows that the model fits the training data better. In the example above, polynomial 9 a the smaller training RMSE than the spline, however, it performs more poorly than the spline in the test data. 

Linear model is underfitting because it performs very poorly in the training data and also in the test data, suggesting it is too simple to capture the relationship well. Polynomial and Spline models have lower RMSE in both training and test, so there is no strong evidence of overfitting. In fact, the spline model performs best on test data, suggesting that its added flexiblity hepls rather than hurts generalization.

# Prediction Variability and Variance

A model can also be evaluated by asking:

> How much would its predictions change if we had obtained a different training sample?

To investigate this, we use the **bootstrap**.

For each bootstrap sample we:
1. Resample the training observations with replacement.
2. Refit each model.
3. Predict bike rentals at the same temperature value.

Large variability in these predictions indicates greater model variance.


In [4]:
X_new = pd.DataFrame({
    "temp": [0.60]
})

B = 500
rng = np.random.default_rng(123)

y_train_array = np.asarray(y_train)
bootstrap_predictions = []

for name, model in models.items():
    for b in range(B):
        idx = rng.choice(
            len(X_train),
            size=len(X_train),
            replace=True
        )

        boot_model = clone(model)

        boot_model.fit(
            X_train.iloc[idx],
            y_train_array[idx]
        )

        prediction = boot_model.predict(X_new).item()

        bootstrap_predictions.append({
            "Model": name,
            "Prediction": prediction
        })

boot_df = pd.DataFrame(bootstrap_predictions)

fig = px.box(
    boot_df,
    x="Model",
    y="Prediction",
    points=False,
    title="Bootstrap Predictions at Temperature = 0.60"
)

fig.show()

variability = (
    boot_df
    .groupby("Model")["Prediction"]
    .agg(["mean", "std"])
    .reset_index()
)

display(
    variability.rename(
        columns={
            "mean": "Mean Prediction",
            "std": "Prediction SD"
        }
    ).round(2)
)


,Model,Mean Prediction,Prediction SD
0,Linear,5242.78,69.88
1,Polynomial 9,5838.97,156.05
2,Spline (7 knots),5976.28,170.01


### What does this tell us about the relationship between model flexibility and sensitivity to the training sample?

At temp = 0.60, the polynomial and spline prediction SDs are 2.23 and 2.43 times the linear model’s, showing that greater flexibility can increase model variance. The spline is slightly more variable, so local flexibility does not guarantee greater stability.

### What does this tell us about the model’s variance and its potential to generalize?

A close training fit with unstable bootstrap predictions suggests high variance and potential overfitting, which can hurt generalization. The polynomial has the lowest training RMSE 1,386.75, but its prediction SD 156.05 is more than twice the linear model’s 69.88. However, high variance does not necessarily mean worse prediction: the polynomial’s test RMSE 1,526.17 is lower than the linear model’s 1,628.52.

# Putting Bias and Variance Together

Increasing flexibility often reduces bias because the model can represent more complicated relationships.
However, greater flexibility can also increase variance because the fitted model may become more sensitive to the particular training sample.

In [5]:
summary = rmse_table.merge(
    variability,
    on="Model"
)

summary = summary.rename(
    columns={
        "mean": "Mean Bootstrap Prediction",
        "std": "Prediction SD"
    }
)

display(summary.round(2))

# Compare test error with prediction variability
fig = px.scatter(
    summary,
    x="Prediction SD",
    y="Test RMSE",
    text="Model",
    size="Prediction SD",
    title="Test Error and Prediction Variability",
    labels={
        "Prediction SD": "Bootstrap Prediction SD",
        "Test RMSE": "Test RMSE"
    }
)

fig.update_traces(
    textposition="top center"
)

fig.show()


,Model,Training RMSE,Test RMSE,Mean Bootstrap Prediction,Prediction SD
0,Linear,1477.19,1628.52,5242.78,69.88
1,Polynomial 9,1386.75,1526.17,5838.97,156.05
2,Spline (7 knots),1388.46,1513.27,5976.28,170.01


Increasing model flexibility can reduce bias but increase variance. Polynomial 9 and Spline add the curved relationship and have lower training RMSE, suggesting lower bias. But their bootstrap predictions indicate higher variance at temp = 0.60. Both have lower test RMSE than Linear model, suggesting that the benefit of lower bias outweighs the cost of higher variance in this split.

### should we simply choose the model with the smallest training RMSE?

The smallest training RMSE only indicates the best fit to the training data, not a good generalization to new data. A flexible model may reduce bias but increase variance and overfit training noise. Therefore, we should also consider test RMSE and bootstrap prediction variability.

### Which of the three models would you carry forward for further evaluation? 

I would carry forward Spline because it has the lowest test RMSE and captures the curved relationship with less extreme boundary behavior than Polynomial 9. This choice is provisional because its test RMSE is just slightly smaller than Polynomial 9. Spline also has the highest bootstrap prediction SD at temp = 0.60, so further validation and stability checks at other temperatures are needed.